# Práctica 1 · Reto: verifiquen una cifra con el activo de su equipo

**Equipo:** *(escriban aquí el nombre del equipo y sus integrantes)*

Este notebook es **opcional**. Repite el ejercicio de la sesión, ahora con el activo que eligió su equipo.

**Entrega:** Jueves 26, 18:00 CDMX time.

## 1 · Los datos de su activo

Escriban el *ticker* de su activo como aparece en Yahoo Finance. Las emisoras de la BMV terminan en `.MX` (por ejemplo `GFNORTEO.MX`). Las fechas quedan fijas para que el resultado no dependa del día en que corran el notebook.

La función de abajo ya está hecha; solo córranla. La primera vez descarga los precios y los guarda en `data/raw`; las siguientes veces lee ese archivo, así todos en el equipo trabajan con los mismos datos. La descarga la estudiaremos a fondo en la sesión 4.

In [3]:

import os
import numpy as np
import pandas as pd

TICKER = "WALMEX.MX"                      # cambien esto por el activo de su equipo
INICIO, FIN = "2016-01-01", "2025-12-31"  # fechas fijas


def cargar_precios_ajustados(ticker, inicio, fin):
    """Devuelve el precio de cierre ajustado de un activo como Serie con índice de fechas.

    Si ya existe data/raw/<ticker>.csv lo lee; si no, descarga de Yahoo Finance y lo guarda.
    """
    archivo = os.path.join("data", "raw", ticker.split(".")[0].lower() + ".csv")
    if os.path.exists(archivo):
        tabla = pd.read_csv(archivo, parse_dates=["fecha"], index_col="fecha")
        return tabla["cierre_ajustado"]

    import yfinance as yf

    descarga = yf.download(ticker, start=inicio, end=fin, auto_adjust=True, progress=False)
    cierre = descarga["Close"]
    if isinstance(cierre, pd.DataFrame):   # algunas versiones devuelven una tabla de una columna
        cierre = cierre.iloc[:, 0]
    cierre = cierre.dropna().rename("cierre_ajustado")
    cierre.index.name = "fecha"
    if cierre.empty:
        raise ValueError(f"Yahoo Finance no devolvió datos para {ticker}. Revisen el ticker.")
    os.makedirs(os.path.dirname(archivo), exist_ok=True)
    cierre.to_csv(archivo)
    return cierre


precios = cargar_precios_ajustados(TICKER, INICIO, FIN)
print(TICKER, "·", precios.index.min().date(), "a", precios.index.max().date(), "·", len(precios), "días")

WALMEX.MX · 2016-01-04 a 2025-12-30 · 2514 días


In [ ]:
# =============================================================================
# LIBRERÍAS QUE USAMOS
# =============================================================================
# pathlib  -> viene incluida con Python. Sirve para manejar rutas de archivos y
#             carpetas como objetos (Path), sin preocuparnos por si la
#             computadora usa "/" (Mac/Linux) o "\" (Windows).
# numpy    -> cálculo numérico: arreglos, logaritmos, raíces, estadísticas.
#             Por convención se importa con el alias "np".
# pandas   -> manejo de datos en tablas (DataFrame) y columnas (Series).
#             Es la que usamos para leer y guardar archivos CSV.
#             Por convención se importa con el alias "pd".
# yfinance -> descarga precios históricos desde Yahoo Finance. Se importa más
#             abajo, dentro de la función, porque solo la necesitamos cuando
#             todavía no tenemos los datos guardados.
# =============================================================================
from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# PARÁMETROS: lo único que cada equipo tiene que modificar
# =============================================================================
# El ticker es la clave con la que Yahoo Finance identifica al activo.
# Las acciones de la Bolsa Mexicana de Valores terminan en ".MX".
TICKER = "WALMEX.MX"                      # cambien esto por el activo de su equipo

# Periodo de análisis en formato "AAAA-MM-DD". Todos los equipos usan las mismas
# fechas para que los resultados sean comparables entre sí.
INICIO, FIN = "2016-01-01", "2025-12-31"  # fechas fijas


# =============================================================================
# UBICAR LA CARPETA DONDE SE GUARDAN LOS DATOS
# =============================================================================
def raiz_del_repo():
    """Sube desde el directorio actual hasta encontrar la raíz del repositorio."""
    # Path.cwd() es la carpeta desde donde se está ejecutando el código.
    # Si corremos el notebook, será .../m1-plantilla-equipo/notebooks/practicas
    # .resolve() la convierte en una ruta absoluta (completa, desde el disco).
    actual = Path.cwd().resolve()

    # actual.parents es la lista de carpetas "hacia arriba": la que contiene a
    # la actual, luego la que contiene a esa, y así hasta llegar al disco.
    # Revisamos primero la carpeta actual y después cada una de las de arriba.
    for carpeta in [actual, *actual.parents]:
        # ¿Cómo sabemos que llegamos a la raíz del repo? Porque ahí viven la
        # carpeta oculta ".git" y el archivo "requirements.txt".
        if (carpeta / ".git").exists() or (carpeta / "requirements.txt").exists():
            return carpeta      # la encontramos: la devolvemos y terminamos

    # Si revisamos todas las carpetas y ninguna era la raíz, avisamos con un
    # error claro en lugar de guardar los datos en un lugar equivocado.
    raise FileNotFoundError("No encontré la raíz del repo (.git o requirements.txt).")


# Con el operador "/" de pathlib vamos armando la ruta carpeta por carpeta.
# Resultado: .../m1-plantilla-equipo/data/raw
# Así los datos siempre quedan en el mismo lugar, sin importar desde qué
# carpeta se ejecute el notebook ni en la computadora de quién.
DATA_RAW = raiz_del_repo() / "data" / "raw"


# =============================================================================
# CARGAR LOS PRECIOS
# =============================================================================
def cargar_precios_ajustados(ticker, inicio, fin):
    """Devuelve el precio de cierre ajustado de un activo como Serie con índice de fechas.

    Si ya existe <raíz del repo>/data/raw/<ticker>.csv lo lee; si no, descarga de
    Yahoo Finance y lo guarda ahí.
    """
    # --- Paso 1: construir el nombre del archivo -----------------------------
    # "WALMEX.MX".split(".") -> ["WALMEX", "MX"]   separa el texto en el punto
    # [0]                    -> "WALMEX"           nos quedamos con la 1a parte
    # .lower()               -> "walmex"           lo pasamos a minúsculas
    # + ".csv"               -> "walmex.csv"       agregamos la extensión
    archivo = DATA_RAW / (ticker.split(".")[0].lower() + ".csv")

    # --- Paso 2: si ya tenemos los datos, los leemos del disco ----------------
    # Así no dependemos de internet cada vez que corremos el notebook y todo el
    # equipo trabaja exactamente con los mismos datos.
    if archivo.exists():
        # parse_dates=["fecha"] -> interpreta esa columna como fechas, no texto
        # index_col="fecha"     -> usa las fechas como índice de la tabla
        tabla = pd.read_csv(archivo, parse_dates=["fecha"], index_col="fecha")
        # Devolvemos solo la columna de precios. El "return" termina la función
        # aquí, así que lo que sigue solo se ejecuta si el archivo no existía.
        return tabla["cierre_ajustado"]

    # --- Paso 3: si no existen, los descargamos de Yahoo Finance --------------
    import yfinance as yf

    # auto_adjust=True -> los precios ya vienen ajustados por dividendos y
    #                     splits. Sin ese ajuste, el día que la empresa paga un
    #                     dividendo parecería una caída de precio, y nuestros
    #                     rendimientos saldrían mal.
    # progress=False   -> no muestra la barra de progreso de la descarga.
    descarga = yf.download(ticker, start=inicio, end=fin, auto_adjust=True, progress=False)

    # La descarga trae varias columnas (Open, High, Low, Close, Volume).
    # Solo necesitamos el precio de cierre.
    cierre = descarga["Close"]

    # Según la versión de yfinance, "Close" puede llegar como tabla de una sola
    # columna en vez de Serie. Si es el caso, tomamos esa única columna:
    # iloc[:, 0] significa "todas las filas, primera columna".
    if isinstance(cierre, pd.DataFrame):
        cierre = cierre.iloc[:, 0]

    # --- Paso 4: limpiar y nombrar --------------------------------------------
    # dropna() -> quita los días sin precio (por ejemplo, días festivos)
    # rename() -> le pone nombre a la columna de precios
    cierre = cierre.dropna().rename("cierre_ajustado")
    # Le ponemos nombre al índice. Tiene que ser "fecha" porque es el nombre
    # que buscamos en el Paso 2 al leer el archivo.
    cierre.index.name = "fecha"

    # --- Paso 5: verificar que sí llegaron datos ------------------------------
    # Si el ticker está mal escrito, Yahoo no marca error: devuelve una tabla
    # vacía. Lo revisamos nosotros para detectar el problema de inmediato.
    if cierre.empty:
        raise ValueError(f"Yahoo Finance no devolvió datos para {ticker}. Revisen el ticker.")

    # --- Paso 6: guardar en disco para la próxima vez -------------------------
    # parents=True  -> crea también las carpetas intermedias si hacen falta
    # exist_ok=True -> no marca error si la carpeta ya existía
    DATA_RAW.mkdir(parents=True, exist_ok=True)
    cierre.to_csv(archivo)
    return cierre


# =============================================================================
# EJECUCIÓN
# =============================================================================
# Llamamos a la función con los parámetros definidos arriba.
# "precios" es una Serie de pandas: una columna de precios con fechas como índice.
precios = cargar_precios_ajustados(TICKER, INICIO, FIN)

# Verificación rápida: confirmamos qué activo cargamos, la primera y la última
# fecha con datos, y cuántos días de operación hay en total.
# .date() muestra solo la fecha, sin la hora.
print(TICKER, "·", precios.index.min().date(), "a", precios.index.max().date(), "·", len(precios), "días")

# Mostramos dónde quedó guardado el archivo para que puedan ubicarlo.
print("Archivo:", DATA_RAW / (TICKER.split(".")[0].lower() + ".csv"))

WALMEX.MX · 2016-01-04 a 2025-12-30 · 2514 días
Archivo: /Users/ilcemedina/Documents/course_finance/m1-plantilla-equipo/data/raw/walmex.csv


## 2 · La cifra del LLM

Pregunten a un LLM por una cifra de su activo que se pueda calcular con precios. Sugerencia:

> ¿Cuál fue la volatilidad anualizada de la acción de *(su emisora)* en 2025?

Escriban la pregunta exacta, el nombre del LLM y la cifra que les dio.

In [ ]:
LLM_USADO = "Claude"            # por ejemplo "ChatGPT", "Gemini", "Claude"
PREGUNTA = "¿Cuál fue la volatilidad anualizada de la acción de WALMEX en 2025?"             # la pregunta exacta que hicieron
CIFRA_LLM = 0.2521          # la cifra en decimales, por ejemplo 0.22

Version = "Opus 5.5"

## 3 · La cifra con datos

Calculen la volatilidad anualizada de 2025: desviación estándar de los rendimientos logarítmicos diarios de 2025, multiplicada por $\sqrt{252}$. Guárdenla en `volatilidad_2025`.

### ¿Qué es la volatilidad?


La **volatilidad** mide qué tanto se mueven los rendimientos de un activo alrededor de su promedio. Es la medida de riesgo más usada en finanzas: un activo con volatilidad alta puede subir o bajar mucho en poco tiempo; uno con volatilidad baja se mueve poco.

**Ojo:** la volatilidad no dice si la acción subió o bajó, solo qué tanto se movió. Una acción puede terminar el año en el mismo precio en que empezó y aun así haber sido muy volátil.

#### Paso 1 · Rendimientos logarítmicos diarios

Si $P_t$ es el precio de cierre ajustado del día $t$, el **rendimiento logarítmico** de ese día es

$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln P_t - \ln P_{t-1}
$$

Para cambios pequeños, $r_t$ es casi igual al cambio porcentual: si el precio sube 1%, $r_t \approx 0.0100$. Usamos logaritmos porque los rendimientos logarítmicos de varios días **se suman**, lo que simplifica el paso 3.

#### Paso 2 · Volatilidad diaria

Con los $n$ rendimientos diarios del periodo, la volatilidad diaria es su desviación estándar muestral:

$$
\bar{r} = \frac{1}{n}\sum_{t=1}^{n} r_t
\qquad\qquad
\sigma_{\text{diaria}} = \sqrt{\frac{1}{n-1}\sum_{t=1}^{n}\left(r_t - \bar{r}\right)^2}
$$

Se divide entre $n-1$ y no entre $n$ porque estamos estimando con una muestra. Es lo que hace por defecto `.std()` de pandas.

#### Paso 3 · Anualizar

$$
\sigma_{\text{anual}} = \sigma_{\text{diaria}} \times \sqrt{252}
$$

**¿Por qué $\sqrt{252}$?** Un año tiene alrededor de 252 días de operación en la bolsa. El rendimiento logarítmico del año es la suma de los 252 rendimientos diarios, y si esos rendimientos son independientes, lo que se suma es la **varianza**, no la desviación estándar:

$$
\text{Var}\left(\sum_{t=1}^{252} r_t\right) = 252\,\sigma_{\text{diaria}}^2
\quad\Longrightarrow\quad
\sigma_{\text{anual}} = \sqrt{252}\;\sigma_{\text{diaria}}
$$

Por eso el riesgo crece con la **raíz** del tiempo y no de forma proporcional.

### ¿Cómo se interpreta?

Supongamos que obtienen una volatilidad anualizada de **25%** ($\sigma_{\text{diaria}} \approx 25\% / \sqrt{252} \approx 1.6\%$).

- **En un día:** si los rendimientos fueran aproximadamente normales, en cerca de 2 de cada 3 días el precio se movería menos de ±1.6%, y en cerca de 19 de cada 20 días, menos de ±3.2%.
- **En un año:** en cerca de 2 de cada 3 años, el rendimiento logarítmico anual quedaría dentro de ±25% alrededor de su promedio.

Como referencia general para acciones:

| Volatilidad anualizada | Lectura |
|---|---|
| menos de 15% | baja |
| 15% a 30% | típica de una empresa grande y estable |
| 30% a 50% | alta |
| más de 50% | muy alta |

### Limitaciones que hay que tener presentes

1. **Mira hacia atrás.** Es una medida histórica: describe cómo se movió la acción en 2025, no cómo se moverá en 2026.
2. **Trata igual las subidas y las bajadas.** Un salto de +5% aumenta la volatilidad igual que una caída de −5%.
3. **Subestima los eventos extremos.** Los rendimientos reales tienen colas más pesadas que la normal, así que los días muy malos ocurren más seguido de lo que sugieren las reglas de arriba.
4. **Depende de las decisiones de cálculo:** el periodo, la frecuencia (diaria, semanal, mensual), si los precios están ajustados por dividendos y splits, y si se usan rendimientos logarítmicos o simples. Por eso una cifra de volatilidad sin estos datos no se puede comprobar.


In [9]:
# INICIO SOLUCIÓN
rendimientos = np.log(precios).diff().dropna()
volatilidad_2025 = rendimientos.loc["2025"].std() * np.sqrt(252)
# FIN SOLUCIÓN
print(f"Volatilidad anualizada de {TICKER} en 2025, con datos: {volatilidad_2025:.2%}")
if CIFRA_LLM is not None:
    print(f"Cifra de {LLM_USADO}: {CIFRA_LLM:.2%} · diferencia: {CIFRA_LLM - volatilidad_2025:+.2%}")

Volatilidad anualizada de WALMEX.MX en 2025, con datos: 28.02%
Cifra de Claude: 25.21% · diferencia: -2.81%


In [10]:
# Comprobación
assert 0.05 < volatilidad_2025 < 1.5, "Revisen el cálculo: una volatilidad anualizada suele estar entre 5% y 150%."
print("El cálculo es razonable.")

El cálculo es razonable.


## 4 · Conclusión

En tres o cuatro renglones: ¿coincidió la cifra del LLM con la de los datos? Si no, ¿a qué creen que se debe la diferencia? ¿Qué le faltó decir al LLM para que su cifra se pudiera comprobar?

<!-- INICIO SOLUCIÓN -->
Respuesta de ejemplo con el activo de la clase (WALMEX): el LLM dio una cifra redonda y no dijo qué fechas, qué tipo de rendimiento ni qué precio usó, así que no se puede reproducir. Nuestra cifra sale de los precios ajustados de 2025, con rendimientos logarítmicos diarios y 252 días por año, y cualquiera puede volver a calcularla con este notebook. Cuando las cifras no coinciden, las causas más comunes son que el LLM use otro periodo, precios sin ajustar, rendimientos mensuales, o que simplemente haya inventado el número. Para poder comprobarla, la respuesta tendría que incluir fuente, periodo, frecuencia y fórmula.
<!-- FIN SOLUCIÓN -->

## Antes de entregar

- [ ] **Entorno de ejecución → Reiniciar sesión y ejecutar todo** corre sin errores.
- [ ] El archivo de `data/raw` quedó guardado en el repositorio del equipo.
- [ ] Declararon en el README qué LLM usaron y para qué.